# NFL Big Data Bowl 2025: GhostFormer Spatial Tracker
This notebook replicates the entire BDB modeling pipeline, natively built and deployable for Google Colab.
Ensure that the  directory is present in your Colab environment with all relevant CSVs (, , ) before running.

### 1. Environment Setup

In [ ]:
!pip install polars torch numpy pytorch-lightning tqdm matplotlib tensorboardX tensorboard

### 2. Preprocessing Data Pipeline
This parses out relative line-of-scrimmage positional normalizations and filters data efficiently into PyTorch Tensor dictionaries (train & test splits).

In [ ]:
import os
import polars as pl
import torch
import numpy as np
from tqdm import tqdm

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

def preprocess_weeks(weeks, split_name):
    print(f"Preprocessing {split_name} (weeks {weeks})...")
    players = pl.read_csv(f"{DATA_DIR}/players.csv", null_values=["NA", "nan", "N/A", "NaN", ""])
    plays = pl.read_csv(f"{DATA_DIR}/plays.csv", null_values=["NA", "nan", "N/A", "NaN", ""])
    
    # Pre-filter players to just what we need
    players_meta = players.select(["nflId", "displayName", "position"]).unique()
    plays_meta = plays.select("gameId", "playId", "defensiveTeam")
    
    all_frames = []
    
    for w in weeks:
        print(f"Loading tracking_week_{w}.csv...")
        if not os.path.exists(f"{DATA_DIR}/tracking_week_{w}.csv"):
            continue
            
        df = pl.read_csv(f"{DATA_DIR}/tracking_week_{w}.csv", null_values=["NA", "nan", "N/A", "NaN", ""])
        
        # Filter football and only keep BEFORE_SNAP
        df = df.filter((pl.col("displayName") != "football") & (pl.col("frameType") == "BEFORE_SNAP"))
        
        # Join plays and players
        df = df.join(plays_meta, on=["gameId", "playId"], how="inner")
        df = df.join(players_meta, on=["nflId", "displayName"], how="left")
        
        # Add isDefense
        df = df.with_columns(
            isDefense=pl.when(pl.col("club") == pl.col("defensiveTeam")).then(pl.lit(1)).otherwise(pl.lit(0))
        )
        
        # Drop players with missing position
        df = df.filter(pl.col("position").is_not_null())
        
        print(f"Computing middle points for week {w}...")
        centers = df.group_by(["gameId", "playId", "frameId", "isDefense"]).agg([
            pl.col("x").mean().alias("mean_x"),
            pl.col("y").mean().alias("mean_y"),
        ])
        
        middle = centers.group_by(["gameId", "playId", "frameId"]).agg([
            pl.col("mean_x").mean().alias("mid_x"),
            pl.col("mean_y").mean().alias("mid_y"),
        ])
        
        df = df.join(middle, on=["gameId", "playId", "frameId"], how="inner")
        
        # Normalize relative to middle point
        df = df.with_columns([
            (pl.col("x") - pl.col("mid_x")).alias("x_norm"),
            (pl.col("y") - pl.col("mid_y")).alias("y_norm"),
        ])
        
        # We need frames to have EXACTLY 22 players
        counts = df.group_by(["gameId", "playId", "frameId"]).len()
        valid_frames = counts.filter(pl.col("len") == 22).select(["gameId", "playId", "frameId"])
        df = df.join(valid_frames, on=["gameId", "playId", "frameId"], how="inner")
        
        # Group to lists
        frames_df = df.group_by(["gameId", "playId", "frameId"]).agg([
            pl.col("nflId"),
            pl.col("position"),
            pl.col("isDefense"),
            pl.col("x_norm").alias("x"),
            pl.col("y_norm").alias("y"),
        ]).to_dicts()
        
        all_frames.extend(frames_df)
        print(f"Week {w} processed: {len(frames_df)} frames added.")

    print(f"Total {split_name} frames: {len(all_frames)}")
    torch.save(all_frames, f"{DATA_DIR}/{split_name}_frames.pt")

# Run manually:
# preprocess_weeks(list(range(1, 9)), "train")
# preprocess_weeks([9], "test")
if False:
    preprocess_weeks(list(range(1, 9)), "train")
    preprocess_weeks([9], "test")


### 3. Dynamic Masked Dataset
Loads the sequential structures and seamlessly interleaves the 11 possible defensive masks dynamically at the loading stage.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

DEFENSE_POSITIONS = ["CB", "S", "FS", "SS", "LB", "MLB", "OLB", "DE", "DT", "NT"]

# 19 positions
POSITIONS = sorted(['C', 'CB', 'DB', 'DE', 'DT', 'FB', 'FS', 'G', 'ILB', 'LB', 'LS', 'MLB', 'NT', 'OLB', 'QB', 'RB', 'S', 'SS', 'T', 'TE', 'WR'])
POS_TO_IDX = {p: i for i, p in enumerate(POSITIONS)}

class MaskedPlayerDataset(Dataset):
    def __init__(self, frames_path):
        """
        frames_path: path to the saved .pt file containing dictionaries.
        """
        self.frames = torch.load(frames_path, weights_only=False)
        self.defense_positions = set(DEFENSE_POSITIONS)
        
        # Flatten dataset: instead of each index being a frame, 
        # each index is a (frame_idx, mask_player_idx).
        # We only augment by masking out the DEFENSIVE players per iteration (meaning 11 variations per frame).
        
        self.samples = []
        for f_idx, frame in enumerate(self.frames):
            is_def = filter(lambda i: frame['isDefense'][i] == 1, range(22))
            for p_idx in is_def:
                self.samples.append((f_idx, p_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        f_idx, mask_idx = self.samples[idx]
        frame = self.frames[f_idx]

        xs = np.array(frame["x"], dtype=np.float32)
        ys = np.array(frame["y"], dtype=np.float32)
        pos = frame["position"]  # list of strings, len 22

        # Build numeric features: [x, y, is_mask]
        numeric = np.stack([xs, ys, np.zeros_like(xs)], axis=-1)  # (22, 3)
        
        # apply mask
        target = np.array([xs[mask_idx], ys[mask_idx]], dtype=np.float32)
        numeric[mask_idx, 0] = 0.0
        numeric[mask_idx, 1] = 0.0
        numeric[mask_idx, 2] = 1.0  # is_mask flag

        # Position ids
        pos_ids = np.array([POS_TO_IDX.get(p, 0) for p in pos], dtype=np.int64)

        return {
            "numeric": torch.from_numpy(numeric),   # (22, 3)
            "pos_ids": torch.from_numpy(pos_ids),   # (22,)
            "mask_idx": torch.tensor(mask_idx, dtype=torch.long),
            "target": torch.from_numpy(target),     # (2,)
        }

def get_dataloader(frames_path, batch_size=64, shuffle=True, num_workers=4):
    ds = MaskedPlayerDataset(frames_path)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)


### 4. Transformer Architectures (GhostFormer)
Instantiates the underlying transformer handling  and  probabilistic spatial predictions optimized by Gaussian NLL.

In [ ]:
import torch
import torch.nn as nn
from pytorch_lightning import LightningModule

torch.set_float32_matmul_precision("medium")

class GhostFormer(nn.Module):
    """Transformer model for predicting masked player position and coordinates."""
    def __init__(
        self,
        num_positions: int = 21,
        pos_emb_dim: int = 16,
        model_dim: int = 64,
        num_layers: int = 3,
        num_heads: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        # numeric = [x, y, is_mask] -> 3 
        self.numeric_dim = 3
        
        self.pos_embedding = nn.Embedding(num_positions, pos_emb_dim)
        
        # Total incoming feature size
        in_dim = self.numeric_dim + pos_emb_dim
        
        self.input_projection = nn.Sequential(
            nn.Linear(in_dim, model_dim),
            nn.LayerNorm(model_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=model_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Decodes coordinate logits (mean_x, mean_y) -> 2 values
        # we can also output log_var_x, log_var_y (4 values total) to get confidence,
        # but let's just do standard regression for now or MDN.
        # User wants "the model's confidence", so we predict 4 values:
        # [mean_x, mean_y, logvar_x, logvar_y]
        self.decoder = nn.Sequential(
            nn.Linear(model_dim, model_dim // 2),
            nn.GELU(),
            nn.LayerNorm(model_dim // 2),
            nn.Linear(model_dim // 2, 4)
        )

    def forward(self, numeric, pos_ids, mask_idx):
        """
        numeric: (B, 22, 3) 
        pos_ids: (B, 22)
        mask_idx: (B,)
        """
        B, seq_len, _ = numeric.shape
        
        # embed position ID
        pos_embs = self.pos_embedding(pos_ids) # (B, 22, pos_emb_dim)
        
        x = torch.cat([numeric, pos_embs], dim=-1) # (B, 22, in_dim)
        x = self.input_projection(x)
        
        x = self.transformer(x) # (B, 22, model_dim)
        
        # We only care about the masked token's representation
        # Gather the representation of the masked token for each item in the batch
        batch_indices = torch.arange(B, device=x.device)
        masked_token_repr = x[batch_indices, mask_idx] # (B, model_dim)
        
        output = self.decoder(masked_token_repr) # (B, 4)
        
        pred_mean = output[:, :2]
        pred_logvar = output[:, 2:]
        return pred_mean, pred_logvar


class GhostFormerLitModel(LightningModule):
    def __init__(
        self,
        learning_rate: float = 1e-3,
        weight_decay: float = 1e-4,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.model = GhostFormer()

    def forward(self, numeric, pos_ids, mask_idx):
        return self.model(numeric, pos_ids, mask_idx)

    def gaussian_nll_loss(self, pred_mean, pred_logvar, target):
        """
        Calculates the Gaussian Negative Log Likelihood Loss.
        Target is [B, 2].
        """
        var = torch.exp(pred_logvar) + 1e-6
        loss = 0.5 * (torch.log(var) + ((target - pred_mean) ** 2) / var)
        return loss.mean()

    def training_step(self, batch, batch_idx):
        numeric = batch["numeric"]
        pos_ids = batch["pos_ids"]
        mask_idx = batch["mask_idx"]
        target = batch["target"]
        
        pred_mean, pred_logvar = self(numeric, pos_ids, mask_idx)
        loss = self.gaussian_nll_loss(pred_mean, pred_logvar, target)
        
        # Also log the pure MSE for interpretability
        mse = nn.functional.mse_loss(pred_mean, target)
        
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_mse", mse, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        numeric = batch["numeric"]
        pos_ids = batch["pos_ids"]
        mask_idx = batch["mask_idx"]
        target = batch["target"]
        
        pred_mean, pred_logvar = self(numeric, pos_ids, mask_idx)
        loss = self.gaussian_nll_loss(pred_mean, pred_logvar, target)
        
        mse = nn.functional.mse_loss(pred_mean, target)
        
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_mse", mse, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.hparams.learning_rate,
            weight_decay=self.hparams.weight_decay,
        )
        # Using OneCycleLR can be very helpful
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"},
        }


### 5. PyTorch Lightning Trainer
Configured routine to process through the augmented set, evaluate, and save checkpoints.

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger
import torch
import warnings




warnings.filterwarnings("ignore")

def main():
    pl.seed_everything(42)

    # We use part of train frames for training, part for val.
    # Since we saved everything together in train_frames.pt, let's load it.
    print("Loading datasets...")
    full_dataset = MaskedPlayerDataset("data/train_frames.pt")
    
    train_size = int(0.9 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_ds, val_ds = torch.utils.data.random_split(full_dataset, [train_size, val_size])
    
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

    model = GhostFormerLitModel(learning_rate=3e-4)

    logger = TensorBoardLogger("tb_logs", name="ghost_former")
    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss",
        dirpath="checkpoints",
        filename="ghost_former-{epoch:02d}-{val_loss:.2f}",
        save_top_k=1,
        mode="min",
    )
    early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=0.00, patience=3, verbose=False, mode="min")

    trainer = pl.Trainer(
        max_epochs=1,  # Short max epochs for quick test, set higher in practice
        logger=logger,
        callbacks=[checkpoint_callback, early_stop_callback],
        accelerator="auto",
        devices=1,
        log_every_n_steps=50,
        val_check_interval=0.5
    )

    print("Starting training...")
    trainer.fit(model, train_loader, val_loader)

# To start training:
# main()
if False:
    main()


### 6. Evaluation on Week 9 Hold-out
Provides aggregate predictive tracking precision logic.

In [ ]:
import torch
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import DataLoader



def evaluate():
    print("Loading test dataset (Week 9)...")
    test_dataset = MaskedPlayerDataset("data/test_frames.pt")
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

    # Assuming we have a saved checkpoint from training. 
    # If not, we will just use an untrained model to demonstrate the pipeline.
    import glob
    ckpt_files = glob.glob("checkpoints/*.ckpt")
    if ckpt_files:
        print(f"Loading model from checkpoint {ckpt_files[0]}")
        model = GhostFormerLitModel.load_from_checkpoint(ckpt_files[0])
    else:
        print("No checkpoint found. Evaluating with untrained model...")
        model = GhostFormerLitModel()
        
    model.eval()
    
    total_mse = 0
    total_mae = 0
    total_nll = 0
    count = 0
    
    print("Evaluating...")
    with torch.no_grad():
        for batch in test_loader:
            numeric = batch["numeric"]
            pos_ids = batch["pos_ids"]
            mask_idx = batch["mask_idx"]
            target = batch["target"]
            
            pred_mean, pred_logvar = model(numeric, pos_ids, mask_idx)
            
            mse = torch.nn.functional.mse_loss(pred_mean, target, reduction='sum').item()
            mae = torch.nn.functional.l1_loss(pred_mean, target, reduction='sum').item()
            
            # NLL metric
            var = torch.exp(pred_logvar) + 1e-6
            nll = 0.5 * (torch.log(var) + ((target - pred_mean) ** 2) / var).sum().item()
            
            total_mse += mse
            total_mae += mae
            total_nll += nll
            count += numeric.size(0) * 2 # 2 coordinates
            
    print("-"*40)
    print("EVALUATION RESULTS (WEEK 9)")
    print(f"Mean Squared Error (MSE): {total_mse/count:.4f}")
    print(f"Mean Absolute Error (MAE): {total_mae/count:.4f}")
    print(f"Gaussian NLL: {total_nll/(count/2):.4f}")
    print("-"*40)
    print("""
    Future Improvements for Spatial/Temporal Classification:
    Instead of continuous regression, the playing field can be discretized into a grid (e.g. 1x1 yard cells).
    This reframes the prediction into a spatial classification problem, which is naturally suited
    for cross-entropy loss and standard autoregressive LM objective formatting.
    Temporal relationships can be captured by adding Temporal self-attention over the sequence of frames
    instead of processing frames independently.
    """)

# evaluate()
if False:
    evaluate()


### 7. Trajectory Visualizations
Extracts the sequence map per chronological play and outputs the variance/prediction tracks onto an MP4 output. (Downloads/Displays directly within Colab).

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.animation import FuncAnimation
import glob
import os
import random




def draw_field(ax):
    """Draws a minimalist football field on the given ax."""
    # Field dimensions (normalized relative to middle)
    # usually plays happen within -30 to 30 yards of middle.
    ax.set_facecolor('#E6F5E6') # Very light, non-distracting green
    ax.set_xlim(-40, 40)
    ax.set_ylim(-30, 30)
    
    # Draw yard lines
    for i in range(-40, 41, 5):
        ax.axvline(i, color='white', alpha=0.5, linestyle='-', zorder=0)
        
    # Draw hash marks
    for i in range(-40, 41, 1):
        ax.plot([i, i], [-1, 1], color='white', alpha=0.3, zorder=0)
        ax.plot([i, i], [29, 31], color='white', alpha=0.3, zorder=0)
        ax.plot([i, i], [-29, -31], color='white', alpha=0.3, zorder=0)

    ax.axis('off')

def get_ellipse(mean_x, mean_y, logvar_x, logvar_y, n_std=2.0, **kwargs):
    """Returns a matplotlib patch for an ellipse representing confidence."""
    var_x = np.exp(logvar_x)
    var_y = np.exp(logvar_y)
    
    width = 2 * n_std * np.sqrt(var_x)
    height = 2 * n_std * np.sqrt(var_y)
    
    return Ellipse(xy=(mean_x, mean_y), width=width, height=height, **kwargs)

def run_visualization():
    print("Loading test frames for visualization...")
    try:
        frames = torch.load("data/test_frames.pt", weights_only=False)
    except FileNotFoundError:
        print("test_frames.pt not found. Ensure preprocess.py has run on week 9.")
        return
        
    if not frames:
        print("No frames loaded.")
        return
        
    # Pick a random play
    # frames are grouped by gameId, playId
    plays = {}
    for i, frame in enumerate(frames):
        key = (frame['gameId'], frame['playId'])
        if key not in plays:
            plays[key] = []
        plays[key].append((i, frame))
        
    # Try to find a play with at least 15 frames BEFORE_SNAP 
    valid_plays = [k for k, v in plays.items() if len(v) >= 15]
    if not valid_plays:
        selected_key = list(plays.keys())[0]
    else:
        selected_key = random.choice(valid_plays)
        
    play_frames = plays[selected_key]
    
    # Sort them by frameId just in case
    # Assuming the data is already sorted by frameId, but to be sure we don't have it explicitly accessible
    # Actually, we can rely on chronological order of the file
    print(f"Visualizing Game {selected_key[0]}, Play {selected_key[1]} ({len(play_frames)} frames)")
    
    # Load Model
    ckpt_files = glob.glob("checkpoints/*.ckpt")
    if ckpt_files:
        model = GhostFormerLitModel.load_from_checkpoint(ckpt_files[0])
    else:
        model = GhostFormerLitModel()
    model.eval()
    
    # Pick a random defensive player to mask throughout the play
    first_frame = play_frames[0][1]
    defenders = [i for i, is_def in enumerate(first_frame['isDefense']) if is_def == 1]
    if not defenders:
        print("No defenders found!")
        return
    mask_target_idx = defenders[0] 
    
    # Pre-compute model predictions and actuals
    history_pred_x = []
    history_pred_y = []
    history_actual_x = []
    history_actual_y = []
    logvars_x = []
    logvars_y = []
    
    # Extract fixed player info for the rest 21
    
    all_player_coords = [] # frame -> [ (x, y, is_def), ... ]
    
    with torch.no_grad():
        for i, (global_idx, frame) in enumerate(play_frames):
            xs = np.array(frame['x'], dtype=np.float32)
            ys = np.array(frame['y'], dtype=np.float32)
            is_defs = frame['isDefense']
            
            # Record actual target
            history_actual_x.append(xs[mask_target_idx])
            history_actual_y.append(ys[mask_target_idx])
            
            # numeric input representation
            numeric = np.stack([xs, ys, np.zeros_like(xs)], axis=-1)
            numeric[mask_target_idx, 0] = 0.0
            numeric[mask_target_idx, 1] = 0.0
            numeric[mask_target_idx, 2] = 1.0
            
            pos_ids = np.array([POS_TO_IDX.get(p, 0) for p in frame['position']], dtype=np.int64)
            
            # Convert to batch size 1
            num_t = torch.from_numpy(numeric).unsqueeze(0)
            pos_t = torch.from_numpy(pos_ids).unsqueeze(0)
            mask_t = torch.tensor([mask_target_idx], dtype=torch.long)
            
            pred_mean, pred_logvar = model(num_t, pos_t, mask_t)
            px, py = pred_mean[0].numpy()
            plvx, plvy = pred_logvar[0].numpy()
            
            history_pred_x.append(px)
            history_pred_y.append(py)
            logvars_x.append(plvx)
            logvars_y.append(plvy)
            
            # Store all other players
            coords = []
            for pidx in range(22):
                if pidx != mask_target_idx:
                    coords.append((xs[pidx], ys[pidx], is_defs[pidx]))
            all_player_coords.append(coords)
            
    # Now animate
    fig, ax = plt.subplots(figsize=(10, 6))
    
    def update(frame_idx):
        ax.clear()
        draw_field(ax)
        
        # Plot other players
        cur_coords = all_player_coords[frame_idx]
        off_x = [c[0] for c in cur_coords if c[2] == 0]
        off_y = [c[1] for c in cur_coords if c[2] == 0]
        def_x = [c[0] for c in cur_coords if c[2] == 1]
        def_y = [c[1] for c in cur_coords if c[2] == 1]
        
        ax.scatter(off_x, off_y, color='red', s=50, label='Offense', edgecolors='white', zorder=3)
        ax.scatter(def_x, def_y, color='blue', s=50, label='Defense', edgecolors='white', zorder=3)
        
        # Plot mask ground truth
        ax.scatter([history_actual_x[frame_idx]], [history_actual_y[frame_idx]], 
                   color='blue', s=80, marker='X', edgecolors='black', label='Target True', zorder=4)
                   
        # Plot prediction trajectory up to frame
        # Draw path
        if frame_idx > 0:
            ax.plot(history_pred_x[:frame_idx+1], history_pred_y[:frame_idx+1], color='orange', linestyle='--', linewidth=2, zorder=2)
            
        cur_px = history_pred_x[frame_idx]
        cur_py = history_pred_y[frame_idx]
        cur_lvx = logvars_x[frame_idx]
        cur_lvy = logvars_y[frame_idx]
        
        # Plot prediction dot
        ax.scatter([cur_px], [cur_py], color='orange', s=80, marker='*', edgecolors='black', label='Predicted', zorder=5)
        
        # Confidence ellipse
        ellipse = get_ellipse(cur_px, cur_py, cur_lvx, cur_lvy, n_std=1.0, 
                              alpha=0.3, color='orange', zorder=1)
        ax.add_patch(ellipse)
        
        # Confidence text indicator (average stdev)
        conf_val = np.sqrt(np.exp(cur_lvx)) + np.sqrt(np.exp(cur_lvy)) # rough measure
        ax.text(0.02, 0.95, f'Confidence (lower var = better): {conf_val:.2f}\nFrame: {frame_idx+1}/{len(play_frames)}', 
                transform=ax.transAxes, color='black', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
                
        # Title
        ax.set_title(f"Play Tracking: Game {selected_key[0]} - Play {selected_key[1]}")
        ax.legend(loc='lower right')
        
    anim = FuncAnimation(fig, update, frames=len(play_frames), interval=100)
    anim.save('visualization.mp4', writer='ffmpeg', fps=10)
    print("Saved visualization to visualization.mp4")

# run_visualization()
if False:
    run_visualization()

from IPython.display import Video
# Video('visualization.mp4', embed=True)